# WineFox LoRA 微调（Colab）

基于 Qwen3.5-0.8B 的酒狐人设 LoRA 微调（基座与 LoRA 分离）。

**架构变更**：
- 基座模型（Qwen3.5-0.8B）与 LoRA adapter **分开保存**
- 对话时启用 LoRA（带酒狐人设）
- 蒸馏时禁用 LoRA（基座模型，避免人设污染蒸馏结果）

**资源**:
- 基座模型：`unsloth/Qwen3.5-0.8B`（HF 在线拉取）
- 数据集：`winefox_dataset.jsonl`（Google Drive 挂载）
- GPU：T4 即可（0.8B + 4bit + LoRA 约需 6GB 显存）

**流程**:
1. 安装依赖 → 2. 挂载 Drive → 3. 加载模型 + LoRA → 4. 注入 system_prompt + 加载数据 → 5. 训练 → 6. 导出 GGUF

**产出**:
- `qwen3.5-0.8b-Q5_K_M.gguf`（基座模型，桌面用）
- `winefox-lora-Q5_K_M.gguf`（LoRA adapter，桌面用）
- `qwen3.5-0.8b-Q4_K_M.gguf`（基座模型，Android 用）
- `winefox-lora-Q4_K_M.gguf`（LoRA adapter，Android 用）

In [ ]:
# Cell 2: 安装依赖（Colab 上约 3-5 分钟）
%pip install --upgrade pip
%pip install "unsloth[cu128] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps trl peft accelerate bitsandbytes
# 验证
import torch
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 数据集路径（请确保 Drive 中有 WineFox/winefox_dataset.jsonl）
DRIVE_DIR = "/content/drive/MyDrive/WineFox"
DATASET_PATH = f"{DRIVE_DIR}/winefox_dataset.jsonl"

import os
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"数据集未找到: {DATASET_PATH}\n"
        f"请将 winefox_dataset.jsonl 上传到 Drive 的 WineFox/ 目录下"
    )
print(f"数据集路径: {DATASET_PATH}")

In [ ]:
# Cell 4: 加载 Qwen3.5-0.8B + LoRA 配置
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name="unsloth/Qwen3.5-0.8B",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# LoRA: 仅训 language + attention + mlp，不训 vision
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print("LoRA 配置完成")

In [ ]:
# Cell 6: 加载数据集
from datasets import Dataset
# 从JSONL文件加载Winefox数据集
dataset = Dataset.from_json("winefox_merged.jsonl")
# 查看数据集的前3条样本
print(dataset[:3])


In [ ]:
# Cell 7: 训练配置 + 启动
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=16,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_ratio=0.05,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        remove_unused_columns=True,
        max_length=2048,
        assistant_only_loss=True,
    ),
)

import time
t0 = time.time()
trainer.train()
print(f"\n训练完成，耗时: {(time.time()-t0)/60:.1f} 分钟")

In [ ]:
# Cell 8: 保存 LoRA adapter（不合并回基座）
# 基座模型与 LoRA 分离：对话时启用 LoRA，蒸馏时禁用 LoRA
lora_output_dir = f"{DRIVE_DIR}/winefox_lora_adapter"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print(f"LoRA adapter 已保存到: {lora_output_dir}")

# Cell 9: 转换 LoRA adapter 为 GGUF 格式
# 需要 llama.cpp 的 convert_lora_to_gguf.py
import subprocess
LLAMA_CPP_DIR = "/content/llama.cpp"  # Colab 中 clone llama.cpp
if not os.path.exists(LLAMA_CPP_DIR):
    subprocess.run(["git", "clone", "https://github.com/ggml-org/llama.cpp", LLAMA_CPP_DIR], check=True)

# 转换 LoRA adapter → GGUF
subprocess.run([
    "python", f"{LLAMA_CPP_DIR}/convert_lora_to_gguf.py",
    "--outfile", f"{DRIVE_DIR}/winefox-lora-f16.gguf",
    lora_output_dir
], check=True)
print("LoRA GGUF 转换完成（f16）")

# 量化为 Q5_K_M 和 Q4_K_M
subprocess.run([
    f"{LLAMA_CPP_DIR}/build/bin/llama-quantize",
    f"{DRIVE_DIR}/winefox-lora-f16.gguf",
    f"{DRIVE_DIR}/winefox-lora-Q5_K_M.gguf", "Q5_K_M"
], check=True)
subprocess.run([
    f"{LLAMA_CPP_DIR}/build/bin/llama-quantize",
    f"{DRIVE_DIR}/winefox-lora-f16.gguf",
    f"{DRIVE_DIR}/winefox-lora-Q4_K_M.gguf", "Q4_K_M"
], check=True)
print("LoRA 量化完成（Q5_K_M + Q4_K_M）")

# Cell 10: 转换基座模型 Qwen3.5-0.8B 为 GGUF 格式（如果尚未转换）
base_gguf = f"{DRIVE_DIR}/qwen3.5-0.8b-Q5_K_M.gguf"
if not os.path.exists(base_gguf):
    subprocess.run([
        "python", f"{LLAMA_CPP_DIR}/convert_hf_to_gguf.py",
        "unsloth/Qwen3.5-0.8B",
        "--outfile", f"{DRIVE_DIR}/qwen3.5-0.8b-f16.gguf",
        "--outtype", "f16"
    ], check=True)
    # 量化
    subprocess.run([
        f"{LLAMA_CPP_DIR}/build/bin/llama-quantize",
        f"{DRIVE_DIR}/qwen3.5-0.8b-f16.gguf",
        base_gguf, "Q5_K_M"
    ], check=True)
    subprocess.run([
        f"{LLAMA_CPP_DIR}/build/bin/llama-quantize",
        f"{DRIVE_DIR}/qwen3.5-0.8b-f16.gguf",
        f"{DRIVE_DIR}/qwen3.5-0.8b-Q4_K_M.gguf", "Q4_K_M"
    ], check=True)
    print("基座模型 GGUF 转换 + 量化完成")
else:
    print(f"基座模型已存在: {base_gguf}")

print("\n=== 全部产出 ===")
print(f"基座（桌面）: {DRIVE_DIR}/qwen3.5-0.8b-Q5_K_M.gguf")
print(f"LoRA（桌面）: {DRIVE_DIR}/winefox-lora-Q5_K_M.gguf")
print(f"基座（Android）: {DRIVE_DIR}/qwen3.5-0.8b-Q4_K_M.gguf")
print(f"LoRA（Android）: {DRIVE_DIR}/winefox-lora-Q4_K_M.gguf")
